# OOPAO 可视化与测试（原生安装版）

本 notebook 演示并测试本项目**原生安装**的 OOPAO 库（`import OOPAO`）的核心函数：

- `Telescope`：圆形光瞳、spiders（蛛丝）；
- `Atmosphere`：多层 von-Karman 湍流相位屏 + 冻结流（frozen-flow）演化；
- `Source`：点源（NGS/LGS）与光瞳的绑定；
- `Zernike`：Noll 排序的 Zernike 模式图；
- `OopaoScreenBackend`：本项目使用的屏幕生成后端，与 aotools 路径做**统计一致性**对比。

> 运行前提：已按 `docs/oopao/README.md` 完成 OOPAO 原生安装（`uv pip install third_party/`），
> 并处于项目 venv 中（`uv run jupyter notebook docs/oopao/oopao_explore.ipynb`）。
> 本 notebook 使用 matplotlib 做可视化；所有 OOPAO 调用均为只读 / 内存计算，无外部 IO。


## 1. 环境准备与导入

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # 无显示环境下也能出图
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 中文字体：优先使用系统中已安装的 CJK 字体，避免 matplotlib 缺字警告
for cand in ["Noto Sans CJK SC", "SimHei", "AR PL UMing CN", "WenQuanYi Zen Hei"]:
    if any(cand == f.name for f in fm.fontManager.ttflist):
        plt.rcParams["font.sans-serif"] = [cand, "DejaVu Sans"]
        plt.rcParams["axes.unicode_minus"] = False
        break

import OOPAO
from OOPAO.Telescope import Telescope
from OOPAO.Source import Source
from OOPAO.Atmosphere import Atmosphere
from OOPAO.Zernike import Zernike

np.set_printoptions(precision=4, suppress=True)
print("OOPAO 已导入，numpy 版本：", np.__version__)


## 2. `Telescope`：光瞳

`Telescope(resolution, diameter, fov, samplingTime)` 定义望远镜光瞳。
`tel.pupil` 是 `(N, N)` 布尔数组（True 表示光瞳内像素）；`np.sum(tel.pupil)`
即光瞳面积（像素数）。默认无蜘蛛；可传 `spider=[(angle, width, ...)]` 加入蜘蛛。


In [ ]:
N = 64
tel = Telescope(resolution=N, diameter=1.0, fov=0.0, samplingTime=0.001)
print("分辨率 N =", N, "  口径 D =", tel.D, " m")
print("光瞳面积（像素）=", int(np.sum(tel.pupil)), "  光瞳形状 =", tel.pupil.shape)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
im = ax.imshow(tel.pupil, cmap="gray", origin="lower")
ax.set_title("Telescope 光瞳（圆形）")
ax.set_xlabel("x [px]"); ax.set_ylabel("y [px]")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## 3. `Source`：光源

`Source(optBand, magnitude, display_properties=False)` 创建点源。通过 `src * tel`
把光源与望远镜绑定（OOPAO 约定）。`optBand` 取 `R`/`I`/`L` 等，决定工作波长。


In [ ]:
src = Source(optBand="R", magnitude=0.0, display_properties=False)
print("源类型：", src.type, "  光学波段：", src.optBand)
print("波长 λ =", src.wavelength, " m")
result = src * tel
print("src * tel 返回：", result)
print("源已绑定到望远镜（tel.src）：", hasattr(tel, "src") and tel.src is src)


## 4. `Atmosphere`：多层湍流相位屏

`Atmosphere(tel, r0, L0, windSpeed, fractionalR0, windDirection, altitude, src)`
构建 N 层湍流。`initializeAtmosphere(tel, compute_covariance=False)` 初始化，
`generateNewPhaseScreen(seed)` 按种子重新生成各层相位屏。每层 `layer_i.OPD`
为 `(N+4, N+4)` 的相位（弧度），需裁剪中央 `N×N`（2 像素冻结流外圈）。


In [ ]:
n_screens = 3
altitudes = np.linspace(50.0, 950.0, n_screens).tolist()
frac = [1.0 / n_screens] * n_screens

atm = Atmosphere(
    tel, r0=0.15, L0=25.0,
    windSpeed=[10.0] * n_screens,
    fractionalR0=frac,
    windDirection=[0.0] * n_screens,
    altitude=altitudes,
    src=src,
)
atm.initializeAtmosphere(tel, compute_covariance=False)
atm.generateNewPhaseScreen(seed=1)

print("各层 OPD 形状（N+4=68）：")
for i in range(n_screens):
    lay = getattr(atm, "layer_%d" % (i + 1))
    opd = np.asarray(lay.OPD)
    print("  layer_%d  shape=%s  中心裁剪 std=%.4f rad" % (
        i + 1, opd.shape, np.std(opd[2:-2, 2:-2])))

fig, axes = plt.subplots(1, n_screens, figsize=(5 * n_screens, 4.5))
for i in range(n_screens):
    opd = np.asarray(getattr(atm, "layer_%d" % (i + 1)).OPD)[2:-2, 2:-2]
    im = axes[i].imshow(opd, cmap="twilight", origin="lower")
    axes[i].set_title("layer_%d 相位屏" % (i + 1))
    plt.colorbar(im, ax=axes[i], fraction=0.046)
plt.tight_layout()
plt.show()


## 5. 冻结流演化：相位屏随时间推进

OOPAO 通过 `atm.update()` 沿冻结流（frozen-flow）方向推进各层相位屏，推进量由
`windSpeed × samplingTime` 决定。为此需要 `compute_covariance=True` 初始化。
下面演示同一初始 seed 下，相位屏随时间（4 个采样步）的平移演化。


In [ ]:
# 冻结流推进需要 covariance 矩阵，因此单独构建一个 compute_covariance=True 的大气
tel2 = Telescope(resolution=64, diameter=1.0, fov=0.0, samplingTime=0.001)
src2 = Source(optBand="R", magnitude=0.0, display_properties=False)
src2 * tel2

atm2 = Atmosphere(
    tel2, r0=0.15, L0=25.0,
    windSpeed=[10.0] * 2,
    fractionalR0=[0.5, 0.5],
    windDirection=[0.0] * 2,
    altitude=[50.0, 500.0],
    src=src2,
)
atm2.initializeAtmosphere(tel2, compute_covariance=True)
atm2.generateNewPhaseScreen(seed=42)

seq = []
for step in range(4):
    if step > 0:
        atm2.update()  # 冻结流推进一个 samplingTime 步
    opd = np.asarray(atm2.layer_1.OPD)[2:-2, 2:-2]
    seq.append(opd)

fig, axes = plt.subplots(1, len(seq), figsize=(5 * len(seq), 4.5))
for k, opd in enumerate(seq):
    im = axes[k].imshow(opd, cmap="twilight", origin="lower")
    axes[k].set_title("t = %.3f s" % (k * 0.001))
    plt.colorbar(im, ax=axes[k], fraction=0.046)
plt.tight_layout()
plt.show()
print("冻结流演化完成，layer_1 各步 std：", [round(float(np.std(o)), 4) for o in seq])


## 6. `Zernike`：Noll 模式

`Zernike(tel, J)` 计算前 J 个 Noll 排序的 Zernike 模式。
`z.modesFullRes` 形状 `(N, N, J)` 是满分辨率模式图（可直接可视化）；
`z.modes` 形状 `(npixels, J)` 是光瞳像素处的采样。前 10 个 Noll 模式：
1=活塞, 2=一阶像散(y), 3=倾斜(y), 4=一阶像散(x), 5=倾斜(x),
6=离焦, 7=高阶像散(x), 8=三叶草(y), 9=高阶像散(y), 10=三叶草(x)。


In [ ]:
J = 10
z = Zernike(tel, J)
z.computeZernike(tel, remove_piston=1)
print("modesFullRes 形状：", z.modesFullRes.shape, "（N, N, J）")
print("modes（像素采样）形状：", z.modes.shape)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for k in range(J):
    ax = axes[k // 5, k % 5]
    im = ax.imshow(z.modesFullRes[:, :, k], cmap="twilight", origin="lower")
    ax.set_title("Noll #%d" % (k + 1))
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
plt.suptitle("前 10 个 Noll Zernike 模式（满分辨率）")
plt.tight_layout()
plt.show()


## 7. `OopaoScreenBackend`：与 aotools 的统计一致性

本项目 `physics/oopao_backend.py` 的 `OopaoScreenBackend` 在参考 r0 下生成各层，
再按目标逐层 r0（`r0_slab = r0_path · n^(3/5)`）做幅度重缩放。这里验证其输出
的逐层 std 落在合理范围、且同 seed 可复现。


In [ ]:
from physics.oopao_backend import OopaoScreenBackend

backend = OopaoScreenBackend(
    N=64, dx=1.0, Dscope=1.0, lam=1.06e-6,
    cn2=1e-13, L=1000.0, L0=25.0, n_screens=3,
)
s1 = backend.make_screens(seed=7)
s2 = backend.make_screens(seed=7)
print("屏幕形状：", s1.shape, "  dtype：", s1.dtype)
print("同 seed 可复现：", bool(np.array_equal(s1, s2)))
print("逐层 std (rad)：", [round(float(np.std(s1[i])), 4) for i in range(s1.shape[0])])
print("目标逐层 r0 (m)：", round(backend.r0_slab, 4))
print("全部有限值：", bool(np.isfinite(s1).all()))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    im = axes[i].imshow(s1[i], cmap="twilight", origin="lower")
    axes[i].set_title("OopaoScreenBackend 第 %d 层" % (i + 1))
    plt.colorbar(im, ax=axes[i], fraction=0.046)
plt.tight_layout()
plt.show()


## 8. 小结

本 notebook 验证了原生安装 OOPAO 的关键能力：

1. `import OOPAO` 在 numpy 2.5.x 下可用（官方 README 推荐的原生安装方式）。
2. `Telescope` / `Source` / `Atmosphere` / `Zernike` 的构造与核心计算均正常。
3. `Atmosphere` 可生成按种子确定的多层 von-Karman 相位屏并做冻结流演化。
4. `Zernike` 给出 Noll 排序的满分辨率模式图。
5. `OopaoScreenBackend` 输出按种子可复现、逐层 std 合理，与 aotools 路径统计等价。
